In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Read the CSV file into a pandas DataFrame
df = pd.read_csv('combined_final_extended.csv')

# Convert 'FRET' column to numeric if it's not already
df['FRET'] = pd.to_numeric(df['FRET'], errors='coerce')

# Define the desired order of x-axis categories - these are examples of all the different desired ordered used, remove the # for the order you prefer
#desired_order = ['RfaH','2','NIH-11','5','6','4S','NIH-9S','2LQ8','7','1','8','10','NIH-13','NusG']  
#desired_order = ['RfaH','16','17','NusG']
#desired_order = ['RfaH','5','6','4','NIH-9','2LQ8','7','NusG']
#desired_order = ['RfaH','5','4','NIH-9','2LQ8','10','NusG']
#desired_order = ['RfaH', 'NIH-9', 'NIH-9S', 'NIH-9L', 'NusG']

# Filter the DataFrame if needed
filtered_df = df[df['Protein'].isin(desired_order)]

# Only use values that actually exist in the data
available_categories = filtered_df['Protein'].unique()
valid_order = [x for x in desired_order if x in available_categories]

# Calculate n counts and build new labels
n_counts = filtered_df['Protein'].value_counts()
new_labels = [f"{protein}\n(n={n_counts[protein]})" for protein in valid_order]

# Set clean white background style
sns.set_style("white")

# adjust the overall size of the figure, this is important for the larger plots because it tries to make them too small and you can't read data
plt.figure(figsize=(12, 6))

#important for noting the color of each variants actual violin in the plot
color_map = {
    'RfaH': 'cyan', '1': 'red', '2': 'cyan', '4': 'cyan', '4S': 'cyan', '4L': 'cyan',
    '5': 'cyan', '6': 'cyan', 'NIH-9': 'cyan', 'NIH-11': 'cyan', 'NIH-9L': 'cyan', 'NIH-9S': 'cyan',
    'NusG': 'red', '7': 'red', '8': 'red', '10': 'red', '10L': 'red', '10S': 'red',
    '2LQ8': 'red', 'NIH-13': 'red', '16': 'grey', '17': 'grey'
}

# Generate a violin plot 
# inner=None removes the default interior violin plot elements to keep your data points clean
# cut=0 keeps the kernel density estimation bound strictly within your actual min/max data range
ax = sns.violinplot(
    x='Protein', 
    y='FRET', 
    data=filtered_df, 
    order=valid_order,
    palette=color_map,
    inner=None,
    cut=1.5
)

# Draw a clean, thick white median line across each violin plot
for i, protein in enumerate(valid_order):
    # Calculate the exact median value for the current protein
    median_val = filtered_df[filtered_df['Protein'] == protein]['FRET'].median()
    
    # Draw a solid horizontal line across the center of each violin plot
    # xmin and xmax define how wide the line segment spreads across the violin width
    ax.hlines(
        y=median_val, 
        xmin=i - 0.25, 
        xmax=i + 0.25, 
        color='white', 
        linewidth=3.5, 
        linestyle='-',
        zorder=3  # Puts the line clearly on top of the violin fill
    )

#  Overlay a swarm/strip plot
sns.stripplot(
    x='Protein', 
    y='FRET', 
    data=filtered_df, 
    order=valid_order, 
    color='black', 
    alpha=0.3, 
    size=3,
    jitter=0.25, 
    ax=ax
)

# Add a horizontal dashed line across the entire plot
ax.axhline(
    y=12.43,                  # <--- Replace 0.5 with your specific FRET efficiency value (FL - 13.25%, T - 12.6%, optT - 12.43%)
    color='black',          # Color of the line
    linestyle='--',         # Dashed line style
    linewidth=1.5,          # Thickness of the line
    zorder=1                # Places it behind the data points but in front of the background
)

# Clean background frame elements
sns.despine()

# Titles and Labels (Increased font sizes and bolded)
plt.title('FRET Efficiency of optTruncated', fontsize=16, fontweight='bold', pad=15)      #change this title depending on which data you are analyzing
plt.xlabel('Variants', fontsize=14, fontweight='bold', labelpad=10)  
plt.ylabel('FRET Efficiency (%)', fontsize=14, fontweight='bold', labelpad=10)  

# Clear default labels and cleanly set horizontal n counts
ax.set_xticks(range(len(valid_order)))
ax.set_xticklabels([f"({n_counts[p]})" for p in valid_order], 
                   fontsize=10, color='dimgray', fontweight='bold')

# Push the horizontal n counts down a bit to leave room for the protein names
ax.tick_params(axis='x', pad=55) 

# Manually place the rotated protein names directly above the horizontal n counts
for i, protein in enumerate(valid_order):
    ax.text(
        x=i,                        # Center text on the respective violin plot index
        y=-3.5,                     # Adjust this negative value to shift it down under the x-axis line
        s=protein,                  # The protein name text
        rotation=45,                # <--- Rotate only the protein name!
        ha='center', va='top',
        fontweight='bold', 
        fontsize=12,
        color='black'
    )

# Make Y-axis tick marks bold cleanly
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels([f"{int(y)}" for y in ax.get_yticks()], fontweight='bold', fontsize=12)

# Set final axis bounds - important to adjust depending on data. 
ax.set_ylim(-2, 30)

# Create Custom Legend handles
legend_elements = [
    Patch(facecolor='cyan', edgecolor='black', label='⍺-helical'),
    Patch(facecolor='red', edgecolor='black', label='β-sheet')
    #Patch(facecolor='grey', edgecolor='black', label='Ambiguous') use this when generating violin plots of 16 and 17
]

# Display custom bold legend placed outside the plot space
leg = ax.legend(
    handles=legend_elements,
    loc='upper left',               # Binds the upper-left corner of the legend box...
    bbox_to_anchor=(0.9, 1),       # ...to just past the right edge (1.02) and at the top (1) of the plot
    ncol=1,  
    framealpha=0.9,
    title='Suggested CTD\nStructure (CD)',
    title_fontsize=14,
    fontsize=12
)
leg.get_title().set_fontweight('bold')

# Save and show the violin plot image
plt.savefig('violin_plot_updatded_ext_final_2026 10 07.png', dpi=1600, bbox_inches='tight')   #adjust this as the saved file name as needed
plt.show()